# 深入 Agent 编排：用 LangGraph 实现 Agentic Workflows

> 本 Notebook 接续 `deep_agent_core_langchain_langgraph.ipynb`。前一份讲清楚 LangChain、LangGraph 与 Deep Agents 的基础；这一份专门解决复杂任务如何编排、约束、验证和回放。

我们最终要实现的不是一个把所有工具都塞进去的万能 Agent，而是一套在不确定处使用模型、在确定处由程序严格控制的工作流。

## 0. 课程来源与学习地图

本 Notebook 的五种编排模式参考吴恩达（Andrew Ng）教授的 *AI Agentic Design Patterns* 课程：

1. **Prompt Chaining**：把固定的大任务拆成小步骤。
2. **Routing**：根据请求类型进入已知的专用流程。
3. **Parallelization**：把互不依赖的检查或子任务并行执行。
4. **Orchestrator-Workers**：由主控动态拆分任务，并分派给 Worker。
5. **Evaluator-Optimizer**：生成、评审、修订，直到满足可执行的质量标准。

这里不是对课程内容的逐字复述，而是把这些模式落实到当前项目所用的 LangGraph，并补上工程中最容易漏掉的状态契约、权限、失败路径、评测和可观测性。

### 学完后应该能回答

- Workflow 和 Agent 的边界是什么？
- 何时用固定图，何时让模型规划或路由？
- 多个 Worker 同时写入 State 时，怎样避免结果覆盖？
- 如何防止动态分工、评审循环和工具重试失控？
- 怎样让 Agent 的建议不能绕过业务权限直接变成生产操作？
- 如何从一次失败任务中回放并定位：是模型、工具、Prompt 还是编排有问题？

## 1. 环境准备

本 Notebook 只使用项目已有的 `langgraph` 和 `pydantic`。所有模型决策由一个确定性 `mock_llm` 代替，所以可以离线运行，不读取 API Key、不访问网络，也不写数据库。

如果要把示例接到真实模型，请将 `mock_llm` 替换为 `llm.with_structured_output(...)`。不要让模型返回一段自由文本后，再靠字符串猜测分类、计划或审批结果。

In [1]:
from __future__ import annotations

import operator
from typing import Annotated, Literal, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Send
from pydantic import BaseModel, Field


def mock_llm(prompt: str) -> str:
    """教学替身：生产环境替换为受约束的结构化模型调用。"""
    return f'[模型草稿] {prompt}'


print('依赖加载完成。所有示例默认离线运行。')

依赖加载完成。所有示例默认离线运行。


## 2. 先决定：这是 Workflow，还是 Agent？

**Workflow** 的节点和边主要由代码预先决定，模型只负责某个受限步骤的语义任务。

**Agent** 在边界内决定下一步工具、执行顺序或子任务划分。它适合步骤本身无法预先穷举的任务，但代价是更难预测、测试和审计。

判断原则：不要因为任务里出现了 LLM 就把整个系统做成 Agent。固定流程优先用 Workflow；只有“不知道下一步是什么”时，才给模型决策权。

| 场景 | 优先方案 | 原因 |
| --- | --- | --- |
| 提取订单号 -> 查询订单 -> 生成回复 | 普通函数或固定 Workflow | 输入、步骤和权限稳定 |
| 根据问题走售后、技术或账务流程 | Routing | 分支有限、可观察 |
| 同时核对事实、敏感字段、格式 | Parallelization | 工作独立，可降低总耗时 |
| 陌生研究问题需要自己拆解角度 | Orchestrator-Workers | 子任务数量与内容事先未知 |
| 报告需要反复补全证据和边界 | Evaluator-Optimizer | 有明确质量标准，可有限修订 |
| 退款、发布、删除或写入生产系统 | 人工审批 + 后端策略 | 模型不能自行放行 |

## 3. State：编排的契约

State 不是把完整对话历史、工具原文和临时想法全塞进去的字典。它是节点之间的接口契约，只保留后续节点实际需要的数据。

建议把状态分为三类：

- **业务事实**：订单 ID、可信工具结果、引用 ID。只允许可信工具或确定性节点写入。
- **控制状态**：路由结果、重试次数、预算、审批状态。只能由编排代码或人工审批节点写入。
- **生成产物**：草稿、摘要、客户回复。模型可以写入，但必须被后续校验。

长文本、原始附件和大规模检索结果应放外部存储；State 保存摘要、引用 ID 或文件路径。

In [3]:
class ReportState(TypedDict, total=False):
    request: str
    category: str
    plan: list[str]
    evidence: Annotated[list[str], operator.add]
    draft: str
    feedback: str
    score: int
    attempts: int
    final: str
    audit_log: Annotated[list[str], operator.add]


def audit(event: str) -> dict:
    return {'audit_log': [event]}


print(ReportState.__annotations__)

{'request': ForwardRef('str', module='__main__', owner=<class '__main__.ReportState'>), 'category': ForwardRef('str', module='__main__', owner=<class '__main__.ReportState'>), 'plan': ForwardRef('list[str]', module='__main__', owner=<class '__main__.ReportState'>), 'evidence': ForwardRef('Annotated[list[str], operator.add]', module='__main__', owner=<class '__main__.ReportState'>), 'draft': ForwardRef('str', module='__main__', owner=<class '__main__.ReportState'>), 'feedback': ForwardRef('str', module='__main__', owner=<class '__main__.ReportState'>), 'score': ForwardRef('int', module='__main__', owner=<class '__main__.ReportState'>), 'attempts': ForwardRef('int', module='__main__', owner=<class '__main__.ReportState'>), 'final': ForwardRef('str', module='__main__', owner=<class '__main__.ReportState'>), 'audit_log': ForwardRef('Annotated[list[str], operator.add]', module='__main__', owner=<class '__main__.ReportState'>)}


### Reducer 为什么重要？

`evidence` 和 `audit_log` 用 `Annotated[..., operator.add]` 定义 reducer。多个并行节点更新这些字段时，LangGraph 会追加结果；没有 reducer 时，多个节点竞争写同一字段会造成覆盖或图定义错误。

而 `attempts`、`category`、`approved` 这类单值控制字段不应该被并行节点同时写入。

# 4. 模式一：Prompt Chaining

Prompt Chaining 将一个困难任务拆成固定步骤。前一个节点的输出成为后一个节点输入，每个中间产物都可以插入结构、长度、敏感字段或引用检查。

```text
request -> extract_constraints -> write_draft -> validate -> final
```

适合：顺序稳定的提取、起草、格式化任务。

不适合：系统不知道接下来要查什么、要拆成多少部分的开放式任务。

In [4]:
class ChainState(TypedDict):
    request: str
    constraints: list[str]
    draft: str
    final: str


def extract_constraints(_: ChainState):
    # 真实实现：使用 Pydantic Schema 限制模型输出。
    return {'constraints': ['中文', '不超过 120 字', '必须说明边界']}


def write_draft(state: ChainState):
    prompt = f"请求：{state['request']}；约束：{state['constraints']}"
    return {'draft': mock_llm(prompt)}


def validate_and_finish(state: ChainState):
    # 确定性规则必须由代码执行，而不是让模型自评。
    return {'final': state['draft'][:120]}


builder = StateGraph(ChainState)
builder.add_node('extract_constraints', extract_constraints)
builder.add_node('write_draft', write_draft)
builder.add_node('validate', validate_and_finish)
builder.add_edge(START, 'extract_constraints')
builder.add_edge('extract_constraints', 'write_draft')
builder.add_edge('write_draft', 'validate')
builder.add_edge('validate', END)
prompt_chain = builder.compile()

chain_result = prompt_chain.invoke({'request': '解释 LangGraph 的价值'})
print(chain_result['final'])
assert chain_result['final'].startswith('[模型草稿]')

[模型草稿] 请求：解释 LangGraph 的价值；约束：['中文', '不超过 120 字', '必须说明边界']


### 哪些地方应设置门槛？

不要在每个节点后都加一次 LLM 评审。只为真实失败模式设置门槛：

| 中间产物 | 应由代码完成的检查 |
| --- | --- |
| 分类标签 | 是否属于允许枚举 |
| JSON / Pydantic 输出 | Schema 是否通过 |
| 引用 | 引用 ID 是否来自本次检索结果 |
| 用户可见文本 | 长度、敏感字段、必填信息 |
| 副作用请求 | 权限、幂等键、审批令牌 |

# 5. 模式二：Routing

Routing 让一个分类节点将请求送进**已知的**专用流程。它的价值不只是效率，更是权限收敛：账务分支不应该突然拿到删除知识库的工具。

```text
                  -> billing_flow ->
incoming -> classify                -> END
                  -> technical_flow ->
                  -> human_handoff ->
```

In [5]:
class RouteState(TypedDict):
    request: str
    category: str
    answer: str


def classify_request(state: RouteState):
    text = state['request']
    if '发票' in text or '退款' in text:
        category = 'billing'
    elif '报错' in text or '登录' in text:
        category = 'technical'
    else:
        category = 'human'  # 未知意图默认不猜。
    return {'category': category}


def billing_flow(_: RouteState):
    return {'answer': '账务流程：查询订单、核验政策，再给出建议。'}


def technical_flow(_: RouteState):
    return {'answer': '技术流程：收集版本、错误信息和复现步骤。'}


def human_handoff(_: RouteState):
    return {'answer': '无法可靠分类，已转人工处理。'}


def choose_route(state: RouteState) -> Literal['billing', 'technical', 'human']:
    return state['category']


builder = StateGraph(RouteState)
builder.add_node('classify', classify_request)
builder.add_node('billing', billing_flow)
builder.add_node('technical', technical_flow)
builder.add_node('human', human_handoff)
builder.add_edge(START, 'classify')
builder.add_conditional_edges('classify', choose_route)
builder.add_edge('billing', END)
builder.add_edge('technical', END)
builder.add_edge('human', END)
router_graph = builder.compile()

route_result = router_graph.invoke({'request': '登录时报错怎么办？', 'category': '', 'answer': ''})
print(route_result)
assert route_result['category'] == 'technical'

{'request': '登录时报错怎么办？', 'category': 'technical', 'answer': '技术流程：收集版本、错误信息和复现步骤。'}


### 路由的工程规则

1. 分类输出使用有限枚举，例如 `billing | technical | human`，不用任意自然语言。
2. 低置信度、冲突意图、敏感意图进入澄清或人工分支。
3. 每个分支只挂载它需要的最小工具集。
4. 记录请求摘要、分类、置信度、模型版本和路由理由，才能复盘错误分类。

真实模型路由时，建议让模型输出 `RouteDecision(category: Literal[...], confidence: float, reason: str)`。然后由代码检查置信度阈值，而不是相信模型写的“我很确定”。

# 6. 模式三：Parallelization

并行只适用于互不依赖的工作。典型场景是同时执行事实核查、风险核查和格式核查。它的目标是降低端到端延迟或提高覆盖率，不是为了让架构看起来像多 Agent。

```text
               -> fact_check ---\
request -> prepare                -> merge -> END
               -> risk_check ---/
               -> style_check --/
```

In [ ]:
class ParallelState(TypedDict):
    request: str
    checks: Annotated[list[str], operator.add]
    final: str


def fact_check(state: ParallelState):
    return {'checks': [f"事实核查：已核对 '{state['request']}' 的来源。"]}


def risk_check(_: ParallelState):
    return {'checks': ['风险核查：未发现需要人工审批的动作。']}


def style_check(_: ParallelState):
    return {'checks': ['表达核查：可以使用简洁中文回复。']}


def merge_checks(state: ParallelState):
    return {'final': '\n'.join(state['checks'])}


builder = StateGraph(ParallelState)
builder.add_node('fact_check', fact_check)
builder.add_node('risk_check', risk_check)
builder.add_node('style_check', style_check)
builder.add_node('merge', merge_checks)
builder.add_edge(START, 'fact_check')
builder.add_edge(START, 'risk_check')
builder.add_edge(START, 'style_check')
builder.add_edge(['fact_check', 'risk_check', 'style_check'], 'merge')
builder.add_edge('merge', END)
parallel_graph = builder.compile()

parallel_result = parallel_graph.invoke({'request': '写一段 LangGraph 简介', 'checks': [], 'final': ''})
print(parallel_result['final'])
assert len(parallel_result['checks']) == 3

### 两种并行方式

| 方式 | 目的 | 如何合并 |
| --- | --- | --- |
| Sectioning | 把任务拆成不同部分，例如技术、成本、风险 | 汇总全部部分，并检查是否缺失 |
| Voting | 多个独立评审检查同一份答案 | 投票、阈值或采用最保守结论 |

并行前问一句：节点 B 是否真的不需要节点 A 的输出？如果答案是否定的，它们就应该串行。

# 7. 模式四：Orchestrator-Workers

当子任务的数量、主题或所需技能无法在写图时确定，才使用 Orchestrator-Workers。例如“为企业知识库 Agent 写调研报告”可能需要从能力边界、成本、风险、集成方式中选取若干角度。

LangGraph 的 `Send` 用于动态 fan-out：Orchestrator 规划后，为每个 section 派发一个 Worker 状态。Worker 完成后，结果通过 reducer 汇合给 Synthesizer。

```text
goal -> orchestrator -> [worker 1, worker 2, ...] -> synthesizer -> END
```

In [ ]:
class OrchestrationState(TypedDict, total=False):
    topic: str
    sections: list[str]
    section: str
    findings: Annotated[list[str], operator.add]
    report: str


def plan_sections(_: OrchestrationState):
    # 真实实现：限制为 Schema Plan(sections: list[str])，再校验最大数量。
    return {'sections': ['能力边界', '工程成本', '风险控制']}


def assign_workers(state: OrchestrationState):
    return [
        Send('research_section', {'topic': state['topic'], 'section': section})
        for section in state['sections']
    ]


def research_section(state: OrchestrationState):
    finding = f"{state['section']}：围绕 {state['topic']} 的核查结论。"
    return {'findings': [finding]}


def synthesize(state: OrchestrationState):
    report = '\n'.join(f'- {item}' for item in state['findings'])
    return {'report': report}


builder = StateGraph(OrchestrationState)
builder.add_node('plan_sections', plan_sections)
builder.add_node('research_section', research_section)
builder.add_node('synthesize', synthesize)
builder.add_edge(START, 'plan_sections')
builder.add_conditional_edges('plan_sections', assign_workers, ['research_section'])
builder.add_edge('research_section', 'synthesize')
builder.add_edge('synthesize', END)
orchestrator_graph = builder.compile()

orchestration_result = orchestrator_graph.invoke({'topic': '企业知识库 Agent', 'findings': []})
print(orchestration_result['report'])
assert len(orchestration_result['findings']) == 3

### 动态编排的四个硬边界

动态拆分最常见的问题不是模型不够聪明，而是没有上限。生产中至少限制：

| 约束 | 示例 |
| --- | --- |
| `max_workers` | 最多 4 个子任务，防止成本指数增长 |
| `allowed_tools` | Worker 只能用与本职相关的只读工具 |
| `deadline` | 超时后返回已完成内容和缺失项 |
| `output_schema` | 每个 Worker 都返回 `findings`、`sources`、`confidence` |

不要让 Worker 无上限地产生 Worker。两层分工可以覆盖大多数业务任务；需要更深层级时，必须显式设计预算、回收和取消机制。

# 8. 模式五：Evaluator-Optimizer

该模式将生成与评审分开：Generator 写草稿，Evaluator 根据明确标准返回问题和修改建议；未通过且尚未达到上限时，流程回到修订节点。

```text
draft -> evaluate -- pass --> final
             |
             +-- revise --> draft
```

适合报告、方案、代码说明、客服回复等“有清晰质量标准，但一次生成未必达标”的工作。

In [ ]:
class ReviewState(TypedDict):
    topic: str
    draft: str
    feedback: str
    score: int
    attempts: int
    final: str


def write_report(state: ReviewState):
    attempt = state['attempts'] + 1
    draft = (
        f"{state['topic']} 的初稿。"
        if attempt == 1
        else f"{state['topic']}：包含结论、证据和适用边界的修订稿。"
    )
    return {'draft': draft, 'attempts': attempt}


def evaluate_report(state: ReviewState):
    required = ['结论', '证据', '边界']
    score = sum(word in state['draft'] for word in required)
    feedback = '补充结论、证据和适用边界。' if score < 3 else '通过'
    return {'score': score, 'feedback': feedback}


def next_after_review(state: ReviewState) -> Literal['finalize', 'revise']:
    # 上限必须由代码强制执行，不能只写在 Prompt 中。
    return 'finalize' if state['score'] == 3 or state['attempts'] >= 2 else 'revise'


def finalize(state: ReviewState):
    suffix = '' if state['score'] == 3 else '\n[已达修订上限，需人工复核。]'
    return {'final': state['draft'] + suffix}


builder = StateGraph(ReviewState)
builder.add_node('write', write_report)
builder.add_node('evaluate', evaluate_report)
builder.add_node('finalize', finalize)
builder.add_edge(START, 'write')
builder.add_edge('write', 'evaluate')
builder.add_conditional_edges('evaluate', next_after_review, {'revise': 'write', 'finalize': 'finalize'})
builder.add_edge('finalize', END)
review_graph = builder.compile()

review_result = review_graph.invoke({
    'topic': 'Agent 编排', 'draft': '', 'feedback': '', 'score': 0, 'attempts': 0, 'final': ''
})
print(review_result['final'])
assert review_result['attempts'] == 2
assert review_result['score'] == 3

### 好的评审标准是什么？

“请检查答案是否好”不是可执行标准。将质量拆为可核对的维度：

| 维度 | 检查方式 |
| --- | --- |
| 完整性 | 每个必填字段是否存在 |
| 真实性 | 关键结论是否对应工具结果或引用 ID |
| 安全性 | 是否包含敏感字段、越权建议或不可执行动作 |
| 可读性 | 长度、语言、模板字段是否满足要求 |
| 业务正确性 | 规则引擎或审批政策是否允许 |

优先由代码检查可确定项目；LLM 评审只处理语义层面的模糊判断，并应返回“问题、建议和证据”，不能只给一个笼统分数。

# 9. 组合：一个真实的研究报告工作流

实际系统往往组合多种模式。下面的图中，简单请求走 Prompt Chaining；复杂请求由 Orchestrator 动态拆分并并行检索；所有生成结果都经过评审、格式校验和权限检查。

```mermaid
flowchart TD
    A[用户请求] --> B[输入校验与风险识别]
    B --> C{Routing}
    C -->|简单问答| D[Prompt Chain]
    C -->|复杂研究| E[Orchestrator]
    E --> F[Workers 并行检索]
    F --> G[Synthesize]
    D --> H[Evaluator]
    G --> H
    H -->|通过| I[格式与权限检查]
    H -->|需修订且未超限| J[Revise]
    J --> H
    H -->|高风险或超限| K[人工复核]
    I --> L[最终结果与审计记录]
```

组合顺序应是：先定义输入、工具和风险边界，再选择最小编排模式；不要先决定“我要做多 Agent”，再为角色寻找任务。

# 10. 工具：系统权限边界

工具不是模型可以随意调用的函数清单，而是业务系统向 Agent 暴露的最小权限接口。良好的工具契约让模型更容易正确调用、下游无需解析随意文本、评测能够核对事实。

读操作至少要校验调用者权限、输入长度和结果数量；写操作还应包含幂等键、审批令牌、审计 ID 与可回滚范围。

In [ ]:
class SearchResult(BaseModel):
    source_id: str
    title: str
    excerpt: str
    confidence: float = Field(ge=0, le=1)


class SearchResponse(BaseModel):
    query: str
    results: list[SearchResult]


def search_knowledge_base(query: str, limit: int = 3) -> SearchResponse:
    """只读工具示例；真实实现还要校验用户身份和 limit 上限。"""
    if not query.strip():
        raise ValueError('query 不能为空')
    if not 1 <= limit <= 10:
        raise ValueError('limit 必须在 1 到 10 之间')
    return SearchResponse(
        query=query,
        results=[SearchResult(
            source_id='kb-101',
            title='LangGraph State 指南',
            excerpt='StateGraph 使用节点和边组织有状态流程。',
            confidence=0.91,
        )],
    )


search_result = search_knowledge_base('LangGraph State')
print(search_result.model_dump())
assert search_result.results[0].source_id == 'kb-101'

### 建议的工具权限矩阵

| 角色 | 允许工具 | 不允许的能力 |
| --- | --- | --- |
| Router | 轻量分类、风险识别 | 业务写入、全量检索 |
| Research Worker | 只读检索、网页或 API 查询 | 发送邮件、修改订单 |
| Synthesizer | 读取已验证证据 | 直接调用生产写接口 |
| Executor | 经批准的单一写工具 | 再规划或扩大权限 |
| Human reviewer | 批准、拒绝、补充说明 | 无限制的系统管理操作 |

System Prompt 可以解释规则，但真正的权限必须由后端身份与策略系统执行。

# 11. 人工审批：建议与执行必须分离

Agent 可以生成“建议退款”或“建议发布”的结论，但不能把这段自然语言当成已经批准。审批状态必须是独立的、受控的字段，且真正写入动作需要后端再次校验。

LangGraph 中可以使用 `interrupt` 暂停图并等待用户恢复。本单元先用确定性门槛展示最小原则：未批准时，执行节点只返回拒绝原因，绝不调用副作用工具。

In [ ]:
class ApprovalState(TypedDict):
    action: str
    approved: bool
    result: str


def execute_if_approved(state: ApprovalState):
    if not state['approved']:
        return {'result': '未获审批：没有执行任何写操作。'}
    # 真实代码此处还要验证审批令牌、调用人、幂等键和当前业务状态。
    return {'result': f"已批准：可提交动作计划 '{state['action']}'。"}


builder = StateGraph(ApprovalState)
builder.add_node('execute', execute_if_approved)
builder.add_edge(START, 'execute')
builder.add_edge('execute', END)
approval_graph = builder.compile()

approval_result = approval_graph.invoke({'action': '退款 100 元', 'approved': False, 'result': ''})
print(approval_result['result'])
assert '没有执行' in approval_result['result']

# 12. 失败要成为图上的路径

网络抖动、429、参数错误、空检索、模型格式错误都不是异常情况，而是 Agent 正常运行时必须处理的状态。节点返回结构化状态，条件边选择继续、有限重试或转人工。

Prompt 中的“不要无限重试”不是控制机制。重试次数、总时长、工具调用数和 token 预算必须由代码层面限制。

In [ ]:
class ToolState(TypedDict):
    retries: int
    tool_status: Literal['ok', 'retryable_error', 'fatal_error']
    tool_result: str


def route_tool_error(state: ToolState) -> Literal['retry', 'escalate', 'continue']:
    if state['tool_status'] == 'ok':
        return 'continue'
    if state['tool_status'] == 'retryable_error' and state['retries'] < 2:
        return 'retry'
    return 'escalate'


assert route_tool_error({'retries': 0, 'tool_status': 'ok', 'tool_result': ''}) == 'continue'
assert route_tool_error({'retries': 1, 'tool_status': 'retryable_error', 'tool_result': ''}) == 'retry'
assert route_tool_error({'retries': 2, 'tool_status': 'retryable_error', 'tool_result': ''}) == 'escalate'
print('失败路径断言通过。')

| 失败类型 | 合适的路径 |
| --- | --- |
| 网络抖动、429、短暂服务异常 | 指数退避的有限重试 |
| 参数无效、无权限、来源不存在 | 不重试，进入澄清或错误说明 |
| 输出未过 Schema | 将具体校验错误交给修订节点，最多一到两次 |
| 高风险、低置信度、预算耗尽 | 立即停止自动执行，转人工 |

# 13. 可观测性、检查点与回放

一次看起来不错的最终回答无法说明 Agent 是否可靠。要定位错误，需要知道它经过了哪些节点、选了哪个路由、调用了什么工具、每一步花了多久、以及在哪个状态被人工拒绝。

至少记录：

```text
trace_id / thread_id / workflow_version / model_version
输入摘要（脱敏）/ 每次路由选择 / 节点耗时
工具名、参数摘要、结果 ID、错误类型
状态快照 / 重试次数 / 人工审批决定
最终输出 / 自动评测结果 / 用户反馈
```

LangGraph 的 checkpointer 可以按 `thread_id` 保存 State。开发时可使用内存实现，生产环境应使用具备访问控制、保留期限和脱敏策略的持久化存储。

In [ ]:
def make_trace_event(
    trace_id: str,
    node: str,
    outcome: str,
    elapsed_ms: int,
) -> dict[str, str | int]:
    """日志应记录摘要和 ID，不直接记录敏感原文或密钥。"""
    return {
        'trace_id': trace_id,
        'node': node,
        'outcome': outcome,
        'elapsed_ms': elapsed_ms,
    }


event = make_trace_event('trace-demo-001', 'router', 'technical', 14)
print(event)
assert event['node'] == 'router'

# 14. 评测：不要只看回答是否通顺

评测应分层。固定 Workflow 首先验证分支、字段和权限；开放式 Agent 再验证工具选择、任务完成、成本与延迟。

| 层级 | 验证内容 | 例子 |
| --- | --- | --- |
| 节点单测 | 纯函数、分类、边界 | “退款”是否始终路由到 billing |
| 图级测试 | 分支、循环上限、审批门槛 | 评审失败两次后是否停止 |
| 端到端评测集 | 质量、成本、延迟、安全性 | 50 条脱敏历史工单的任务完成率 |

核心指标通常包括：正确分支率、必填字段率、工具调用有效率、违规率、人工转交率、平均延迟和单任务成本。

In [ ]:
route_cases = [
    ('我要申请退款', 'billing'),
    ('登录页面报错', 'technical'),
    ('帮我写一首诗', 'human'),
]

for request, expected in route_cases:
    result = router_graph.invoke({'request': request, 'category': '', 'answer': ''})
    assert result['category'] == expected, (request, result)

print(f'路由回归样本通过：{len(route_cases)} 条')

# 15. 从 Notebook 走向项目：落地清单

在接入真实模型、工具或业务系统之前，逐项确认：

- [ ] 哪些步骤是确定性的？为什么还需要模型？
- [ ] State 是否区分了事实、控制字段和生成物？是否保存了不必要的大文本或敏感信息？
- [ ] 每条条件边是否有默认分支、低置信度分支和循环上限？
- [ ] 每个 Agent / Worker 的工具权限是否最小化？
- [ ] 所有写操作是否有身份验证、审批、幂等键和审计记录？
- [ ] 工具结果、模型输出和最终交付物是否通过各自的 Schema 校验？
- [ ] 超时、限流、空结果、格式错误、预算耗尽时，图会如何流转？
- [ ] 是否能凭 trace 和 State 快照回放一个失败任务？
- [ ] 是否拥有稳定的脱敏样本集，用于修改 Prompt、模型或图后的回归？

最可靠的 Agent 系统通常不是最会自主思考的系统，而是能在不确定处灵活处理、在确定处严格受控、失败时清楚停下的系统。

# 16. 练习

1. 将 `classify_request` 替换为结构化 LLM 路由，加入 `confidence`，低于阈值时转人工。
2. 给 `research_section` 增加 `sources` 与 `confidence`，在 `synthesize` 前过滤没有来源的结论。
3. 把 `review_graph` 的关键词检查换成 Pydantic Schema 和一个 LLM 语义评审节点，保留两次上限。
4. 使用 `InMemorySaver` 为其中一个图加入 `thread_id`，观察多轮调用的 State 变化。
5. 选择项目中的一个真实只读工具，设计它的输入 Schema、输出 Schema、权限范围和失败路径。

# 17. 继续阅读

- Andrew Ng, *AI Agentic Design Patterns*：五种 Workflow 模式的课程来源。
- LangGraph Documentation：Graph API、State reducers、conditional edges、`Send`、persistence、human-in-the-loop。
- 本项目的 `docs/learn_agent_1/deep_agent_core_langchain_langgraph.ipynb`：LangChain、LangGraph、Deep Agents 基础与综合示例。

下一步建议：先逐格运行本 Notebook 的离线示例，再只挑 Routing 或 Evaluator-Optimizer 中一个模式接入真实模型。不要在没有评测样本和权限边界的前提下直接搭建多 Agent。